# 量子畳み込みニューラルネットワーク (QCNN)

**量子畳み込み(quanvolutional)**層は、古典的な畳み込みと同じように働きますが、画像の上をスライドするフィルタが重み行列ではなく、小さなパラメトリック量子回路である点が違います: 各画像パッチを数量子ビットに符号化し、もつれさせ、学習可能な回転を通したあと、単一の期待値として読み出します — これはちょうど、古典的な畳み込みカーネルが1パッチあたり1つの出力値を生成するのと同じ役割を、量子力学的に計算しているだけです。

このノートブックでは、これをゼロから作り、小さな画像分類器(MNIST、数字0と1)の最初の層として学習させます。使うのはPyTorchの通常の`loss.backward()`だけです — blueqatSDKの回路が普通の微分可能なPyTorchテンソルを返すため、特別な勾配計算のトリックは一切必要ありません。

参考文献: Henderson et al., ["Quanvolutional Neural Networks: Powering Image Recognition with Quantum Circuits"](https://arxiv.org/abs/1904.04767) (2019)

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK torchvision

  Cloning https://github.com/blueqat/blueqatSDK to /private/var/folders/cd/spczq0r91lnfn5n14s3mfvxr0000gn/T/pip-req-build-v38m6orb
  Running command git clone --filter=blob:none --quiet https://github.com/blueqat/blueqatSDK /private/var/folders/cd/spczq0r91lnfn5n14s3mfvxr0000gn/T/pip-req-build-v38m6orb


  Resolved https://github.com/blueqat/blueqatSDK to commit fde32d1a168d32fe2da3e0ab886c796095e4e552


  Installing build dependencies ... -

 \ done


  Getting requirements to build wheel ... -

 done


  Preparing metadata (pyproject.toml) ... -

 done


## データ: MNIST、数字0と1

このノートブックの実行時間を現実的な範囲に保つため(量子畳み込みの各パッチは行列の掛け算ではなく、実際の小さな回路シミュレーションです)、2値のサブセット(数字0と1のみ)と少数の学習・テスト画像を使います。全10桁・フルデータセットの場合も全く同じ方法で動きますが、各パッチごとに回路を実行する必要があるため遅くなるだけです。

In [2]:
import torch
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

full_train = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
full_test = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_indices = [i for i, (_, y) in enumerate(full_train) if y in (0, 1)][:200]
test_indices = [i for i, (_, y) in enumerate(full_test) if y in (0, 1)][:100]

trainset = torch.utils.data.Subset(full_train, train_indices)
testset = torch.utils.data.Subset(full_test, test_indices)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=4, shuffle=False)

print(f"train images: {len(trainset)}, test images: {len(testset)}, batches/epoch: {len(trainloader)}")

train images: 200, test images: 100, batches/epoch: 50


## 量子畳み込み層

各 $2\times2$ の画像パッチ(4ピクセル、4量子ビット)について:

1. **符号化**: 各ピクセル値 $x$ を量子ビットの回転にします。$R_y(\arctan x)$ に続いて $R_z(\arctan x^2)$ — これは、範囲に制限のない古典的な値を、範囲の決まった回転角にマッピングする標準的な方法です。
2. **もつれ**: 4量子ビットすべてを環状に結合する`CX`ゲート — これにより、以下の読み出しがピクセル1個だけでなく、パッチ全体に依存するようになります。
3. **学習可能な回転**: 各量子ビットに一般的な単一量子ビット回転`u(theta, phi, lam)`を適用し、`theta`/`phi`/`lam`を学習パラメータとします — これは畳み込みカーネルの学習された重みに相当する、量子版のパラメータです。
4. **読み出し**: 量子ビット0の $Z$ 期待値が、このパッチの単一の出力値になります — 古典的な畳み込みカーネルが1パッチあたり1つの出力値を生成するのとまったく同じです。

これを重複なしですべての $2\times2$ パッチにスライドさせることで、$28\times28$ の画像が $14\times14$ の1チャンネル出力に変換されます — 本物の(量子)畳み込みです。

In [3]:
import math
from blueqat import Circuit
from blueqat.utils import Z
import torch.nn as nn


class Qconv(nn.Module):
    """A single-channel, 2x2, stride-2 quantum convolution layer."""

    def __init__(self, kernel_h=2, kernel_w=2):
        super().__init__()
        self.kernel_h = kernel_h
        self.kernel_w = kernel_w
        self.n_qubit = kernel_h * kernel_w
        self.params = nn.Parameter(torch.rand(self.n_qubit * 3) * 2 * math.pi)

    def _run_patch(self, patch):
        flat = patch.reshape(-1)
        circ = Circuit(self.n_qubit)
        for i in range(self.n_qubit):
            circ.ry(torch.arctan(flat[i]))[i]
            circ.rz(torch.arctan(flat[i] ** 2))[i]
        for i in range(self.n_qubit):
            circ.cx[i, (i + 1) % self.n_qubit]
        for i in range(self.n_qubit):
            circ.u(self.params[3 * i], self.params[3 * i + 1], self.params[3 * i + 2])[i]
        return circ.run(hamiltonian=Z[0])

    def forward(self, x):
        # x: (batch, 1, H, W)
        b, _, h, w = x.shape
        out_h, out_w = h // self.kernel_h, w // self.kernel_w
        out = torch.zeros(b, 1, out_h, out_w)
        for bi in range(b):
            for i in range(out_h):
                for j in range(out_w):
                    patch = x[bi, 0,
                              i * self.kernel_h:(i + 1) * self.kernel_h,
                              j * self.kernel_w:(j + 1) * self.kernel_w]
                    out[bi, 0, i, j] = self._run_patch(patch)
        return out

簡単な健全性チェック: 1パッチをこの層に通し、通常の`.backward()`呼び出しだけでそのパラメータまで勾配が伝わることを確認します — カスタムの`autograd.Function`も、有限差分による勾配推定も不要です。これは、blueqatSDKの回路がずっと微分可能なPyTorchテンソルを返す([100_quantum_classical_hybrid.ipynb](100_quantum_classical_hybrid.ipynb)などの以前のチュートリアルで使われている通り)ことの直接的な帰結です。以前の(旧blueqat SDK上で動かした)同じアイデアの量子畳み込み実装では、代わりに手書きの有限差分による逆伝播が必要でした。回路の出力がまだ微分可能なテンソルではなかったためです。

In [4]:
qconv = Qconv()
sample_patch = torch.rand(2, 2)
output = qconv._run_patch(sample_patch)
output.backward()
print("output:", output.item())
print("gradient reached the circuit's own parameters:", qconv.params.grad is not None)

output: 0.595013048584224
gradient reached the circuit's own parameters: True


## モデル全体

`Qconv`は、`nn.Conv2d`とまったく同じように、普通のPyTorchモデルに組み込めます: $28\times28 \to$ (量子畳み込み) $\to 14\times14 \to$ (最大値プーリング) $\to 7\times7 \to$ (平坦化+線形層) $\to$ 2クラスのスコア。

In [5]:
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.qconv = Qconv(2, 2)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(7 * 7, 2)

    def forward(self, x):
        x = self.qconv(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


net = Net()
print(net)

Net(
  (qconv): Qconv()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=49, out_features=2, bias=True)
)


## 学習

短い学習ループです — 量子に特化した部分はなく、普通のPyTorchです。`loss.backward()`を呼ぶたびに、順伝播中に実行されたすべての量子回路を透過的に微分します。

In [6]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.1)

t0 = time.time()
for epoch in range(2):
    running_loss = 0.0
    for inputs, labels in trainloader:
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"epoch {epoch + 1}: avg loss = {running_loss / len(trainloader):.4f}")

print(f"training time: {time.time() - t0:.1f}s")

epoch 1: avg loss = 0.3798


epoch 2: avg loss = 0.0423
training time: 138.9s


In [7]:
correct = 0
total = 0
with torch.no_grad():
    for images, labels in testloader:
        outputs = net(images)
        predicted = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"test accuracy: {100 * correct / total:.1f}% ({correct}/{total})")

test accuracy: 99.0% (99/100)


学習画像わずか200枚、2エポックだけでも、この小さな量子畳み込みモデルはすでに偶然よりもずっと高い精度で0と1を分離できるはずです(これは比較的簡単な2値分類タスクです)。このノートブックの目的は、古典的なCNNと精度で競うことではありません — 数百枚の画像と単一の4量子ビットパッチフィルタでは、それは無理です — 目的は、通常の`nn.Module`を組み込むのとまったく同じように、通常の誤差逆伝播で最初から最後まで学習できる、本物の量子・古典ハイブリッド畳み込み層を示すことです。

ここからの自然な次のステップは、このノートブックの元になった技術とまったく同じです: 出力チャネルを増やす(古典的な`Conv2d(in_channels, out_channels, ...)`のように、複数の独立した`Qconv`フィルタを使う)、2つ目の量子畳み込み層を重ねる、あるいは全10桁分類タスクへのスケールアップ(上記のコードはどちらも変更不要です — データセット/モデルのサイズを変えるだけです)。